In [1]:
import librosa
import numpy as np
import os
from skimage.feature import local_binary_pattern


def extract_mfcc(y, sr, n_mfcc=80, n_fft=512, hop_length=3, win_length=2, n_mels=128):
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=win_length * sr,
        hop_length=hop_length * sr,
        n_mels=n_mels,
        win_length=win_length * sr
    )

    if mfcc.shape[1] < 9: return None

    # delta = librosa.feature.delta(mfcc)

    # # delta-delta
    # delta2 = librosa.feature.delta(mfcc, order=2)

    # # ghép feature
    # feat_all = np.vstack([mfcc, delta, delta2]).T
    # CMVN
    return mfcc.T   # (num_frames, n_mfcc)


In [2]:
import numpy as np
from skimage.feature import graycomatrix, graycoprops

def clbp(image, P=8, R=1):
    """
    image : 2D array (MFCC / spectrogram)
    P : số neighbor
    R : radius
    return:
        CLBP_S, CLBP_M, CLBP_C
    """
    h, w = image.shape

    # padding
    pad = R
    img = np.pad(image, pad, mode='edge')

    # center
    center = img[pad:pad+h, pad:pad+w]

    # init
    CLBP_S = np.zeros((h, w), dtype=np.uint8)
    CLBP_M = np.zeros((h, w), dtype=np.uint8)

    # magnitude threshold
    diff_list = []

    for p in range(P):
        theta = 2 * np.pi * p / P
        dy = int(round(R * np.sin(theta)))
        dx = int(round(R * np.cos(theta)))

        neighbor = img[pad+dy:pad+dy+h, pad+dx:pad+dx+w]

        diff = neighbor - center
        diff_list.append(np.abs(diff))

    # threshold cho magnitude
    diff_stack = np.stack(diff_list, axis=0)
    T = diff_stack.mean()

    for p in range(P):
        theta = 2 * np.pi * p / P
        dy = int(round(R * np.sin(theta)))
        dx = int(round(R * np.cos(theta)))

        neighbor = img[pad+dy:pad+dy+h, pad+dx:pad+dx+w]
        diff = neighbor - center

        # CLBP_S
        CLBP_S |= ((diff >= 0).astype(np.uint8) << p)

        # CLBP_M
        CLBP_M |= ((np.abs(diff) >= T).astype(np.uint8) << p)

    # CLBP_C
    CLBP_C = (center >= center.mean()).astype(np.uint8)

    return CLBP_S, CLBP_M, CLBP_C

def hist(x, bins=256):
    h, _ = np.histogram(x.ravel(), bins=bins, range=(0, bins))
    return h

def glcm_features(img):
    glcm = graycomatrix(
        img,
        distances=[1, 2],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256,
        symmetric=True,
        normed=True
    )

    feats = []
    for p in ['contrast','energy','homogeneity','correlation']:
        feats.extend(graycoprops(glcm, p).ravel())

    return np.array(feats)


In [3]:
import numpy as np
import librosa
from skimage.feature import local_binary_pattern

def build_mfcc_dataset(
    wav_list,
    sr=44100,
    segment_sec=2,
    out_path="X_mfcc.npy"
):
    all_feats = []

    segment_len = int(segment_sec * sr)

    for wav_path in wav_list:
        print("Processing:", wav_path)

        y, sr = librosa.load(wav_path, sr=sr)

        # nếu audio < 2s
        if len(y) < segment_len:
            segments = [y]
        else:
            num_segments = len(y) // segment_len
            segments = [
                y[i*segment_len:(i+1)*segment_len]
                for i in range(num_segments)
            ]

        for segment in segments:
            mfcc = extract_mfcc(segment, sr)
            print(mfcc.shape)
            if mfcc is None: continue
            CLBP_S, CLBP_M, CLBP_C = clbp(mfcc, P=8, R=1)
            f1 = glcm_features(CLBP_S)
            f2 = glcm_features(CLBP_M)

            feature = np.concatenate([f1, f2])
            all_feats.append(feature)

        print(f"{wav_path} -> {len(segments)} segments")

    X = np.vstack(all_feats)

    np.save(out_path, X)

    print("Saved:", out_path, X.shape)

    return X

In [ ]:
import os
CLASS = "bee"

DIR = f"E:/Pythonfile/Voice-Activity-Detect/data/raw/nuhive_processed/{CLASS}"

wav_list = [
    os.path.join(DIR, f)
    for f in os.listdir(DIR)
    if f.endswith(".wav")
]
print(wav_list)
X_mfcc = build_mfcc_dataset(
    wav_list,
    out_path=f"E:/Pythonfile/Voice-Activity-Detect/data/feature_bee/mfcc_clbp_glcm_{CLASS}_80_nodelta.npy"
)

['E:/Pythonfile/Voice-Activity-Detect/data/raw/nuhive_processed/noqueen\\CF001 - Missing Queen - Day --0-0.wav', 'E:/Pythonfile/Voice-Activity-Detect/data/raw/nuhive_processed/noqueen\\CF001 - Missing Queen - Day --0-1.wav', 'E:/Pythonfile/Voice-Activity-Detect/data/raw/nuhive_processed/noqueen\\CF001 - Missing Queen - Day --0-2.wav', 'E:/Pythonfile/Voice-Activity-Detect/data/raw/nuhive_processed/noqueen\\CF001 - Missing Queen - Day --0-3.wav', 'E:/Pythonfile/Voice-Activity-Detect/data/raw/nuhive_processed/noqueen\\CF001 - Missing Queen - Day --0-4.wav', 'E:/Pythonfile/Voice-Activity-Detect/data/raw/nuhive_processed/noqueen\\CF001 - Missing Queen - Day --2-0.wav', 'E:/Pythonfile/Voice-Activity-Detect/data/raw/nuhive_processed/noqueen\\CJ001 - Missing Queen - Day -  (100)-0-0.wav', 'E:/Pythonfile/Voice-Activity-Detect/data/raw/nuhive_processed/noqueen\\CJ001 - Missing Queen - Day -  (100)-10-0.wav', 'E:/Pythonfile/Voice-Activity-Detect/data/raw/nuhive_processed/noqueen\\CJ001 - Missing 

ValueError: The maximum grayscale value in the image should be smaller than the number of levels.